# Data breaches and their impact  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 5.1.d: Score data breach severity

**Skill:** Analyze Risk · **Maps to:** 5.1.C · **Language:** Python 3 (pytest)

## Problem
Build the severity scorer a security team's intake triage tool would run on each incoming breach
report. Given the categories of data exposed, the number of records affected, and how many days
it took to detect the breach, compute a total severity score and a label.

## Scoring rubric
Start at `score = 0`.

**Data category points** (sum over all categories present in the report, each counted once):
- `"pii"` (personally identifiable information): +40
- `"financial"`: +40
- `"health"`: +40
- `"auth_plaintext"` (plaintext passwords/secrets): +40
- `"auth_hashed"` (hashed/salted passwords): +15
- `"email"`: +10
- `"public"`: +0

**Scale points** (based on `record_count`):
- `< 100`: +0
- `100`-`9,999`: +10
- `10,000`-`999,999`: +20
- `>= 1,000,000`: +30

**Detection-speed points** (based on `days_to_detect`):
- `<= 1`: +0
- `2`-`30`: +10
- `> 30`: +20

## Label thresholds (applied to the total score)
- `score >= 80` -> `"critical"`
- `score >= 50` -> `"high"`
- `score >= 20` -> `"medium"`
- else -> `"low"`

## Function contract
Implement `solve(report: dict) -> dict` in the code editor (see the Reference section below).

## Input
`report`: `{"categories": list[str], "record_count": int, "days_to_detect": int}`

## Output
`{"score": int, "label": str}`

## Examples
```
{"categories": ["public"], "record_count": 50, "days_to_detect": 1}
    -> {"score": 0, "label": "low"}

{"categories": ["pii", "auth_plaintext"], "record_count": 500000, "days_to_detect": 45}
    -> {"score": 40 + 40 + 20 + 20, "label": "critical"}   # = 120

{"categories": ["email"], "record_count": 200, "days_to_detect": 5}
    -> {"score": 10 + 10 + 10, "label": "medium"}          # = 30
```

## Constraints
- Deterministic, offline, standard library only.
- Multiple categories are additive; unknown category strings contribute 0 points.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(report: dict) -> dict
```

### Input (`report`)
`{"categories": list[str], "record_count": int, "days_to_detect": int}`

- `categories`: zero or more of `{"pii", "financial", "health", "auth_plaintext", "auth_hashed",
  "email", "public"}`. Unknown strings are ignored (contribute 0 points).
- `record_count`: non-negative int.
- `days_to_detect`: non-negative int.

### Output (`result`)
`{"score": int, "label": str}` where `label` is one of `"low"`, `"medium"`, `"high"`, `"critical"`.

### Rubric
- Category points (sum, each category counted once): pii=40, financial=40, health=40,
  auth_plaintext=40, auth_hashed=15, email=10, public=0, unknown=0.
- Scale points: `<100`->0, `100-9999`->10, `10000-999999`->20, `>=1000000`->30.
- Detection points: `<=1`->0, `2-30`->10, `>30`->20.
- Label thresholds on total score: `>=80`->critical, `>=50`->high, `>=20`->medium, else low.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Categories with duplicates in the input list are only counted once.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 5.1.d. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import Dict

CATEGORY_POINTS = {
    "pii": 40,
    "financial": 40,
    "health": 40,
    "auth_plaintext": 40,
    "auth_hashed": 15,
    "email": 10,
    "public": 0,
}


def _scale_points(record_count: int) -> int:
    if record_count >= 1_000_000:
        return 30
    if record_count >= 10_000:
        return 20
    if record_count >= 100:
        return 10
    return 0


def _detect_points(days_to_detect: int) -> int:
    if days_to_detect > 30:
        return 20
    if days_to_detect >= 2:
        return 10
    return 0


def _label(score: int) -> str:
    if score >= 80:
        return "critical"
    if score >= 50:
        return "high"
    if score >= 20:
        return "medium"
    return "low"


def solve(report: Dict[str, object]) -> Dict[str, object]:
    """Score a data breach report's severity per the rubric."""
    categories = report.get("categories", [])
    record_count = report.get("record_count", 0)
    days_to_detect = report.get("days_to_detect", 0)

    score = sum(CATEGORY_POINTS.get(cat, 0) for cat in categories)
    score += _scale_points(record_count)
    score += _detect_points(days_to_detect)

    return {"score": score, "label": _label(score)}


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_low_public_small():
    report = {"categories": ["public"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_critical_pii_plaintext_large_slow():
    report = {"categories": ["pii", "auth_plaintext"], "record_count": 500000, "days_to_detect": 45}
    assert impl.solve(report) == {"score": 120, "label": "critical"}


def test_medium_email_small_moderate():
    report = {"categories": ["email"], "record_count": 200, "days_to_detect": 5}
    assert impl.solve(report) == {"score": 30, "label": "medium"}


def test_hashed_auth_only():
    report = {"categories": ["auth_hashed"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 15, "label": "low"}


# ---- hidden / edge cases -------------------------------------------------
def test_high_financial_million_records_fast_detect():
    report = {"categories": ["financial"], "record_count": 2_000_000, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 70, "label": "high"}


def test_multiple_categories_additive():
    report = {"categories": ["pii", "financial", "health"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 120, "label": "critical"}


def test_unknown_category_contributes_zero():
    report = {"categories": ["mystery_category"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_empty_report():
    report = {"categories": [], "record_count": 0, "days_to_detect": 0}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_boundary_record_count_10000():
    report = {"categories": [], "record_count": 10000, "days_to_detect": 0}
    assert impl.solve(report) == {"score": 20, "label": "medium"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
